# Study 10 — Clustering Robustness (Visual Analysis)

Visualizza cluster state: embedding space, cluster coherence, top clusters.

In [ ]:
import os, sqlite3, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
import sqlite_vec  # LOAD sqlite-vec module

# Auto-navigate to repo root
notebook_dir = Path.cwd()
while not (notebook_dir / 'data/db/pathosphere.db').exists():
    notebook_dir = notebook_dir.parent
    if notebook_dir == notebook_dir.parent: break
os.chdir(notebook_dir)

DB = Path('data/db/pathosphere.db').resolve()
CONN = sqlite3.connect(str(DB))
sqlite_vec.load(CONN)  # Register vec0 module
CONN.row_factory = sqlite3.Row
print(f'Connected: {DB}')
print('✓ sqlite-vec loaded')

In [ ]:
# Load RSS clustering data
rss_events = CONN.execute('''
    SELECT e.id, e.title, COUNT(ed.document_id) as size
    FROM events e LEFT JOIN event_documents ed ON e.id = ed.event_id
    WHERE e.origin IN ("rss","comtrade") GROUP BY e.id ORDER BY size DESC
''').fetchall()

# Load embeddings for all RSS docs
all_docs = CONN.execute('''
    SELECT DISTINCT ed.document_id, ed.event_id
    FROM event_documents ed
    JOIN raw_documents r ON ed.document_id = r.id
    WHERE r.origin IN ("rss","comtrade")
''').fetchall()

doc_to_event = {d['document_id']: d['event_id'] for d in all_docs}
doc_ids = list(doc_to_event.keys())

# Load embeddings
embeddings = []
valid_doc_ids = []
for doc_id in doc_ids:
    vec_row = CONN.execute('SELECT embedding FROM vec_documents WHERE document_id = ?', (doc_id,)).fetchone()
    if vec_row and vec_row['embedding']:
        emb = np.frombuffer(vec_row['embedding'], dtype=np.float32).copy()
        embeddings.append(emb)
        valid_doc_ids.append(doc_id)

embeddings = np.array(embeddings)
print(f'Loaded {len(embeddings)} embeddings (dim={embeddings.shape[1]})')
print(f'RSS events: {len(rss_events)}')
print(f'Docs assigned: {len(valid_doc_ids)}')

In [ ]:
# 1. PCA embedding space
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

colors = [hash(doc_to_event[doc_id]) % 20 for doc_id in valid_doc_ids]

fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=colors, cmap='tab20', alpha=0.6, s=30)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('RSS Embedding Space (PCA) — each color is a cluster')
ax.grid(alpha=0.3)
plt.savefig('study_10_embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ study_10_embedding_space.png')

In [ ]:
# 2. Cluster size distribution
sizes = [e['size'] for e in rss_events]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(sizes, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
ax1.set_xlabel('Docs per Cluster')
ax1.set_ylabel('Frequency')
ax1.set_title('Cluster Size Distribution (Linear)')
ax1.grid(alpha=0.3, axis='y')

ax2.hist(sizes, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
ax2.set_xlabel('Docs per Cluster')
ax2.set_ylabel('Frequency')
ax2.set_yscale('log')
ax2.set_title('Cluster Size Distribution (Log) — check for chain-collapse')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('study_10_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ study_10_sizes.png')

In [ ]:
# 3. Top 10 clusters table
fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('tight')
ax.axis('off')

table_data = []
for i, e in enumerate(rss_events[:10], 1):
    is_garbage = e['title'].startswith('||') if e['title'] else False
    status = '❌ GARBAGE' if is_garbage else '✓ READABLE'
    table_data.append([f"{i}", f"{e['size']} docs", status, e['title'][:80]])

table = ax.table(cellText=table_data,
                colLabels=['Rank', 'Size', 'Status', 'Title'],
                cellLoc='left', loc='center',
                colWidths=[0.05, 0.1, 0.15, 0.7])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

for i in range(1, len(table_data) + 1):
    if '❌' in table_data[i-1][2]:
        table[(i, 2)].set_facecolor('#ffcccc')
    else:
        table[(i, 2)].set_facecolor('#ccffcc')

plt.title('Top 10 Clusters by Size', fontsize=14, pad=20)
plt.savefig('study_10_top_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ study_10_top_clusters.png')

In [ ]:
# 4. Coherence analysis
large_events = [e for e in rss_events if e['size'] >= 5][:30]
coherence_data = []

for event in large_events:
    doc_ids_in_cluster = CONN.execute('''
        SELECT DISTINCT ed.document_id FROM event_documents ed
        WHERE ed.event_id = ?
    ''', (event['id'],)).fetchall()
    
    cluster_embeddings = []
    for d in doc_ids_in_cluster:
        if d['document_id'] in valid_doc_ids:
            idx = valid_doc_ids.index(d['document_id'])
            cluster_embeddings.append(embeddings[idx])
    
    if len(cluster_embeddings) >= 2:
        cluster_embeddings = np.array(cluster_embeddings)
        centroid = cluster_embeddings.mean(axis=0)
        dists = [np.linalg.norm(e - centroid) for e in cluster_embeddings]
        avg_sim = 1 - np.mean(dists)**2 / 2
        coherence_data.append({'size': event['size'], 'coherence': avg_sim})

if coherence_data:
    df_coh = pd.DataFrame(coherence_data).sort_values('coherence')
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(range(len(df_coh)), df_coh['coherence'], color='steelblue')
    ax.axvline(0.88, color='red', linestyle='--', linewidth=2, label='Threshold (0.88)')
    ax.set_xlabel('Avg Cosine Similarity to Centroid')
    ax.set_title(f'Coherence: {len(df_coh)} Large Clusters (≥5 docs)')
    ax.legend()
    ax.grid(alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('study_10_coherence.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Coherence: mean={df_coh["coherence"].mean():.3f}, min={df_coh["coherence"].min():.3f}')
    print('✓ study_10_coherence.png')

In [ ]:
# VERDICT
garbage = CONN.execute("SELECT COUNT(*) as c FROM events WHERE title LIKE '||%'").fetchone()['c']
singleton_pct = 100*sum(1 for s in sizes if s == 1)/len(sizes) if sizes else 0

print(f'''
╔════════════════════════════════════════╗
║ CLUSTERING VERDICT                    ║
╚════════════════════════════════════════╝

RSS Events: {len(rss_events)}
Singleton Rate: {singleton_pct:.1f}%
Mean Cluster Size: {np.mean(sizes):.2f}
GDELT Garbage: {garbage}
Coherence (large clusters): {df_coh["coherence"].mean():.3f} (threshold 0.88)

Status: {'SOLID ✓' if garbage == 0 and singleton_pct >= 80 else 'CHECK MANUALLY'}

Plots:
  1. study_10_embedding_space.png
  2. study_10_sizes.png
  3. study_10_top_clusters.png
  4. study_10_coherence.png
''')